# Develop and test a quantum error correction scheme with `qdk.ec`

Taking a quantum error correction scheme from a paper to a declarative artifact is
hard. The checks, readouts, and circuit semantics all have to stay consistent as
the design changes.

`qdk.ec` closes that gap around one artifact: a **qodec**. A qodec is a declarative
description of a compilation pipeline together with the error correction schemes
that lower each layer of it. Because it is *just data*, the same artifact can move
from analysis into a compilation pipeline without a second representation.

This notebook walks the stages the package is organised around:

| stage | module | question it answers |
| --- | --- | --- |
| develop | `qdk.ec` | how do I load, save, and finish a qodec? |
| profile | `qdk.ec.action`, `qdk.ec.checks`, `qdk.ec.distance` | what does this qodec do? |
| test | `qdk.ec.equivalence`, `qdk.ec.lint` | is that what I intended? |

## Installing

`qdk.ec` is an optional extra of the `qdk` package:

```bash
pip install "qdk[ec]"
```

## 1. Develop — load a qodec

`qdk.ec` holds the primitives that move qodecs between disk, memory, and
YAML text. We start from `c4.qodec.yaml`, sitting next to this notebook: the
[[4,2,2]] error-*detecting* code, which encodes two logical qubits in four
physical ones and can detect (but not correct) any single-qubit fault.

In [ ]:
import qdk.ec as ec
from qdk.ec import action, checks, distance, equivalence, lint, readouts

qodec = ec.load_yaml("c4.qodec.yaml")
print(qodec.summary())

A qodec is a chain of **layers**, from the most abstract instruction set down to
the most concrete. Each layer carries the **gadgets** that lower one of its
instructions into a circuit over the layer below. Here there is a single lowering
edge: the logical `C4` instruction set down to physical `stim` operations.

In [ ]:
layer = qodec.layers[0]
print("lowering:", layer.isa.name, "->", qodec.layers[1].isa.name)
print("gadgets: ", sorted(layer.gadgets))


## 2. Profile — characterise the code

`qdk.ec` computes focused, typed characteristics of qodec objects through one
module per question — `action`, `checks`, `code`, `distance`, `faults`,
`readouts`. Start
with the code itself: its stabilizers, its logical operators, and its distance.

In [ ]:
code = qodec.codes["C4"]

print("stabilizers:", list(code.stabilizers))
print("logical X:  ", list(code.x))
print("logical Z:  ", list(code.z))

distance, witness = distance.code_distance_of(code)
print(f"distance:    {distance}  (witness: {[str(p) for p in witness]})")


Distance 2 is exactly what "error *detecting*" means: there is a weight-2 logical
error, so a single fault is always visible but never correctable.

### Declared vs. realized action

Every gadget makes a promise — the action of the instruction it `implements` — and
keeps it with a circuit. Those are two independent objects, and `qdk.ec` can
compute both and compare them. This is the check that catches a transcription slip
between the paper and the circuit.

In [ ]:
measure_zz = layer.gadgets["measure_zz"]

print("declared:", action.declared_action_of(measure_zz))
print("realized:", action.realized_action_of(measure_zz))
print("mismatch:", action.gadget_action_mismatch(measure_zz) or "none")

### Checks and readouts

A gadget's circuit produces raw measurement outcomes. Two derived structures give
those outcomes meaning:

* **checks** — parities of outcomes that are *deterministic*, so a flip signals a
  fault. These are what a decoder consumes.
* **readouts** — the parities that carry the logical answer the instruction
  promised.

Both are discovered by exact simulation, so you never have to derive them by
hand.

In [ ]:
discovered = readouts.profile_of(measure_zz)

print("checks:     ", discovered.checks)
print("observables:", discovered.observables)
print("essential:  ", checks.essential_checks_of(measure_zz))

## 3. Develop — let the tooling finish the draft

Because checks and readouts are *derivable*, an author should not have to write
them. `ec.complete_gadget` fills them in for one gadget, and
`ec.complete_qodec` does it for an entire qodec.

To show it working, take a gadget, throw its checks away, and ask `qdk.ec` to put
them back.

In [ ]:
import qodec as qc

draft = qc.Gadget(
    measure_zz.implements,
    measure_zz.circuit,
    inputs=list(measure_zz.inputs),
    outputs=list(measure_zz.outputs),
    checks=[],
    readouts=[[str(atom) for atom in entry] for entry in measure_zz.readouts],
)
print("draft checks:    ", list(draft.checks))

completed = ec.complete_gadget(draft)
print("completed checks:", [[str(atom) for atom in check] for check in completed.checks])


`complete_qodec` applies the same treatment to every gadget of every layer, and
returns a new qodec — the input is never mutated.

In [ ]:
completed_qodec = ec.complete_qodec(qodec)

for mnemonic, gadget in sorted(completed_qodec.layers[0].gadgets.items()):
    print(f"{mnemonic:16s} {len(gadget.checks)} check(s)")


### Round-tripping through YAML

A qodec is data, so it round-trips. `to_yaml` / `from_yaml` keep it in memory;
`save` / `load` put it on disk. This is the handoff to the compilation pipeline:
the artifact you just tested *is* the deployment config.

In [ ]:
text = ec.to_yaml(completed_qodec)
print(f"{len(text)} characters of YAML, {len(text.splitlines())} lines")

reloaded = ec.from_yaml(text)
print("round-trips:", reloaded.name == completed_qodec.name)


## 4. Test — audit the qodec

`qdk.ec.lint` runs a rule set over the whole qodec and returns structured
diagnostics: each one names the rule that fired, the object it fired on, and why.
This is the "did I write what I meant?" pass.

In [ ]:
report = lint.diagnose(qodec)
print(f"{len(report.errors())} error(s), {len(report.warnings())} warning(s)")

for diagnostic in report.errors() + report.warnings()[:2]:
    print()
    print(f"[{diagnostic.severity.name}] {diagnostic.rule}")
    print(f"  {diagnostic.summary}")


The report flags two kinds of problem here, and both are the kind that is
invisible in a paper and fatal in a pipeline: `measure_xx` declares readout
parities that its own circuit does not produce, and several gadgets never declare
a sign for their output stabilizers, so a decoder cannot tell which frame it is
being handed.

### Equivalence

The other half of testing is comparison: is this refactored gadget the same as the
one I trust? `qdk.ec.equivalence` answers that, and explains a "no".

In [ ]:
measure_xx = layer.gadgets["measure_xx"]

print("measure_zz == itself:    ", equivalence.gadgets_equivalent(measure_zz, measure_zz))
print("measure_zz == measure_xx:", equivalence.gadgets_equivalent(measure_zz, measure_xx))
print("why not:", equivalence.why_not_equivalent(measure_zz, measure_xx))

## Where to go next

* `qdk.ec` provides `load_yaml`, `save_yaml`, `complete_gadget`,
  `complete_qodec`, and `qodec_from_code`.
* `qdk.ec.action`, `.checks`, `.code`, `.distance`, `.faults`, and `.readouts`
  provide one profiling module per question.
* `qdk.ec.equivalence` and `qdk.ec.lint` verify that a qodec does what you
  intended.

The qodec you finish here is ordinary data that can be handed to a downstream
compilation pipeline without another representation.